# 03 — Baseline Models and Evaluation

This notebook runs the repository's three baseline models through the same train-only fitting and validation-evaluation harness. It compares pooled classification quality, calibration, and daily cross-sectional ranking without reading locked-test rows.

The dates and tickers are illustrative. Adjust them to the history available in your local bronze database.

In [ ]:
from dataclasses import replace
from datetime import date
from pathlib import Path

PROVIDER = "yfinance"
TICKERS = ("AAK.ST", "SAAB-B.ST", "VOLV-B.ST")
DATA_CUTOFF = date(2025, 12, 31)

TRAIN_START = date(2010, 1, 1)
TRAIN_END = date(2021, 12, 31)
VALIDATION_START = date(2022, 1, 1)
VALIDATION_END = date(2023, 12, 31)
TEST_START = date(2024, 1, 1)
TEST_END = date(2025, 12, 31)

## Define the common experiment boundary

All three baselines use the same feature set, target, universe, cutoff, and split. Only `ModelSpec` changes. The V2 target set also contains `forward_return_5d`, which is used as a continuous diagnostic outcome; it does not change the ATR-barrier classification target.

In [ ]:
from swingtrader.data.features import DEFAULT_FEATURE_SET
from swingtrader.modeling.datasets import UniverseSpec, V2_PRIMARY_TASK, V2_TARGET_SET
from swingtrader.modeling.experiments import (
    ExperimentSpec,
    ModelSpec,
    TemporalSplitSpec,
)
from swingtrader.modeling.training import (
    CONSTANT_PRIOR_MODEL_TYPE,
    LOGISTIC_REGRESSION_MODEL_TYPE,
    RANDOM_RANKING_MODEL_TYPE,
)

feature_set = DEFAULT_FEATURE_SET.select(
    "returns",
    "trend",
    "volatility",
    "volume",
    name="notebook_ohlcv_features",
    version="1",
)
universe = UniverseSpec(
    name="notebook_training_universe",
    version="1",
    provider=PROVIDER,
    tickers=TICKERS,
)
split_spec = TemporalSplitSpec(
    name="notebook_fixed_holdout",
    version="1",
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    validation_start=VALIDATION_START,
    validation_end=VALIDATION_END,
    test_start=TEST_START,
    test_end=TEST_END,
)

logistic_spec = ModelSpec(
    name="regularized_logistic_regression",
    version="1",
    model_type=LOGISTIC_REGRESSION_MODEL_TYPE,
    hyperparameters={
        "regularization_strength": 1.0,
        "max_iter": 1_000,
        "tolerance": 1e-8,
    },
)
experiment_spec = ExperimentSpec(
    name="notebook_v2_baselines",
    version="1",
    feature_set=feature_set,
    target_set=V2_TARGET_SET,
    task=V2_PRIMARY_TASK,
    universe=universe,
    data_cutoff=DATA_CUTOFF,
    split=split_spec,
    model=logistic_spec,
    random_seeds={"model": 42, "evaluation": 43},
)

## Build and split the canonical dataset

Dataset construction and splitting remain repository-owned. The model harness receives the aligned bundle and positional split result; it does not reconstruct rows independently.

In [ ]:
from swingtrader.data.db import resolve_database_engine
from swingtrader.modeling.datasets import build_temporal_dataset
from swingtrader.modeling.experiments import FixedTemporalSplitter

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
database_url = f"sqlite+pysqlite:///{(repo_root / 'data' / 'swingtrader.sqlite').as_posix()}"
engine = resolve_database_engine(database_url=database_url)

bundle = build_temporal_dataset(engine=engine, spec=experiment_spec.dataset_spec)
split_result = FixedTemporalSplitter(experiment_spec.split).assign(bundle)

{
    "dataset_digest": bundle.manifest.digest,
    "split_digest": split_result.manifest.digest,
    "train": split_result.summary("train").to_manifest(),
    "validation": split_result.summary("validation").to_manifest(),
}

## Run the three baselines

`run_baseline_experiment()` fits only on train and evaluates validation by default. The local artifact directory is separate for each model, while every report uses the same schema and evaluation settings.

In [ ]:
from swingtrader.modeling.training import EvaluationConfig, run_baseline_experiment

model_specs = {
    "constant_prior": ModelSpec(
        name="constant_prior",
        version="1",
        model_type=CONSTANT_PRIOR_MODEL_TYPE,
    ),
    "random_ranking": ModelSpec(
        name="date_matched_random_ranking",
        version="1",
        model_type=RANDOM_RANKING_MODEL_TYPE,
    ),
    "logistic": logistic_spec,
}
config = EvaluationConfig(
    classification_threshold=0.5,
    calibration_bins=10,
    score_quantiles=10,
    top_k=5,
    random_seed=43,
)

results = {}
for name, model_spec in model_specs.items():
    candidate = replace(
        experiment_spec,
        name=f"notebook_v2_{name}",
        model=model_spec,
    )
    results[name] = run_baseline_experiment(
        bundle,
        split_result,
        candidate,
        ranking_return_column="forward_return_5d",
        evaluation_config=config,
        artifact_directory=repo_root / "artifacts" / "notebook_baselines" / name,
    )

list(results)

## Compare validation metrics

The constant prior is the probability baseline, random ranking is the date-matched selection baseline, and logistic regression is the first learned benchmark. A later nonlinear model should be compared against all three.

In [ ]:
import pandas as pd

comparison = pd.DataFrame(
    {
        name: result.reports["validation"].aggregate_metrics
        for name, result in results.items()
    }
).T
comparison[
    [
        "average_precision",
        "roc_auc",
        "log_loss",
        "brier_score",
        "mean_daily_spearman",
        "mean_top_k_return",
        "mean_random_top_k_return",
        "top_k_return_lift",
    ]
]

## Inspect calibration and ranking tables

Quantiles are assigned within each trading date. The aggregate quantile table gives equal weight to each represented date rather than allowing dates with larger candidate universes to dominate.

In [ ]:
logistic_validation = results["logistic"].reports["validation"]

logistic_validation.calibration

In [ ]:
logistic_validation.score_quantiles

In [ ]:
logistic_validation.per_date_metrics[
    [
        "trading_date",
        "roc_auc",
        "average_precision",
        "spearman",
        "top_k_positive_rate",
        "top_k_mean_return",
        "random_top_k_mean_return",
    ]
].tail()

## Inspect retained fitted state and artifacts

The logistic manifest contains the train-fitted medians and scales together with coefficients and optimizer diagnostics. Validation rows never influence this state.

In [ ]:
results["logistic"].model.to_manifest()

In [ ]:
artifact_root = repo_root / "artifacts" / "notebook_baselines" / "logistic"
sorted(path.relative_to(artifact_root).as_posix() for path in artifact_root.rglob("*"))

## Locked-test evaluation

Leave the switch below false while comparing features, models, hyperparameters, classification thresholds, or ranking rules. Once those choices are frozen, set it to true for a final evaluation. Changing a decision after seeing the test result means the test is no longer locked.

In [ ]:
RUN_LOCKED_TEST = False

if RUN_LOCKED_TEST:
    final_result = run_baseline_experiment(
        bundle,
        split_result,
        experiment_spec,
        ranking_return_column="forward_return_5d",
        evaluation_config=config,
        include_locked_test=True,
        artifact_directory=repo_root / "artifacts" / "notebook_baselines" / "final",
    )
    final_result.reports["test"].aggregate_metrics
else:
    print("Locked test not evaluated.")